# Field-To-Curve Outputs

Exploratory setup for the first field-input to curve-output smoke task. The production run should use `p2-DisorderML/HPC/FieldToCurveOutputs/A0-HPC_FieldToCurve-test.py`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from resources.MLdata import DATA
from resources.MLmetrics import curve_performance_diagnostics, print_curve_diagnostics, plot_curve_diagnostics


In [ ]:
DATA_ROOT = Path("Z:/p1/data/Ti/disNodes/0.2/FCC")
TASK = "UT"
MODEL_TYPE = "TR"
NSIMS = 256
SPLIT_SEED = 42
PCA_COMPONENTS = 16
FIELD_INPUT_CONFIG = {
    "components": ("U1", "U2"),
    "drop_frame0": True,
    "layout": "auto",
}

RUN_TINY_LOCAL_TRAIN = False


In [ ]:
DAT = DATA(
    path=DATA_ROOT,
    load=True,
    split_frac=0.9,
    split_seed=SPLIT_SEED,
    range_split=(True, False),
    LAT="FCC",
    dis="disNodes",
    dN=0.2,
    d_data="in",
    mechMode=TASK,
    nsims=NSIMS,
    model=MODEL_TYPE,
    input_kind="field",
    output_kind="curve",
    field_input_config=FIELD_INPUT_CONFIG,
    scale=("symm", "inout"),
    reduce_dim=("PCA", "out", None, PCA_COMPONENTS, True),
    round_decimals=5,
    geom_feats=(True, True),
)

summary = pd.DataFrame(
    [
        ("field input shape", DAT.UT_field_input_shape),
        ("train tokens", DAT.UT_train_in.shape),
        ("val tokens", DAT.UT_val_in.shape),
        ("test tokens", DAT.UT_test_in.shape),
        ("latent train targets", DAT.UT_train_out.shape),
        ("node feature count", DAT.UT_train_in.shape[-1]),
    ],
    columns=["item", "value"],
)
display(summary)
print("First node feature names:", DAT.UT_node_feature_names[:12])


In [ ]:
pca = DAT.UT_OUTreducer
print(f"PCA explained variance with {PCA_COMPONENTS} components: {np.sum(pca.explained_variance_ratio_):.6f}")

oracle = DAT.UT_OUTreconstructor(DAT.UT_test_out)
truth = DAT.UT_test_out_df.to_numpy(dtype=float)
train_truth = DAT.UT_train_out_df.to_numpy(dtype=float)
diagnostics = curve_performance_diagnostics(
    oracle,
    truth,
    x_values=DAT.UT_OUT_df,
    train_truth=train_truth,
    mode="ut",
)
print_curve_diagnostics(diagnostics, label="UT PCA oracle")
plot_curve_diagnostics(DAT.UT_OUT_df, oracle, truth=truth, diagnostics=diagnostics, mode="ut", max_samples=24)


In [ ]:
if RUN_TINY_LOCAL_TRAIN:
    import torch
    import torch.nn as nn

    from resources.MLfunc import EarlyStopping
    from resources.MLmodels import MODEL, Transformer

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    in_shape = DAT.UT_train_in.shape[1:]
    model = Transformer(
        in_size=int(in_shape[-1]),
        seq_len=int(in_shape[-2]),
        h_size=[128],
        out_size=int(DAT.UT_train_out.shape[-1]),
        d_model=128,
        n_heads=4,
        n_layers=2,
        ff_mult=4,
        act="gelu",
        encoder_act="gelu",
        norm="layer",
        dropout=0.1,
        att_dropout=0.1,
        head_norm="layer",
        head_dropout=0.05,
        pool="mean",
        use_cls_token=False,
    ).to(device)

    MOD = MODEL(
        typ=DAT.model,
        model=model,
        lossf=nn.MSELoss(reduction="mean"),
        opt=("adamw", 1e-5),
        batch=4,
        lr=2e-4,
        data=DAT,
        mechMode=DAT.mechMode,
        scheduler=("plateau", "min", 0.5, 5, 1e-4),
        earlyStop=EarlyStopping(patience=10, min_delta=1e-5, verbose=True),
        w_init="auto",
        device=device,
        scan_matches_on_init=False,
    )
    MOD.train(n_epochs=3, verbose=1, plot=True)
    MOD.evaluate_split("test", diagnostics=True, diag_plot=True)
